In [ ]:
            print(f"\n✅ '{output_fz}' 파일이 생성되었습니다. 파일 탐색기에서 다운로드 후 Fritzing에서 수정하세요!")

import cv2
import numpy as np
import glob
import os
import base64
import json
from ultralytics import YOLO
from google.colab import files
from IPython.display import HTML, display

# --- 1. 배경 이미지 (깔끔한 빵판) 로드 ---
CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'
cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)

if cb_canvas is None:
    print("🚨 [오류] 'clean_breadboard.png' 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드 해주세요.")
else:
    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape

    # 여백 보정 (기본적인 시작점)
    roi_x1, roi_x2 = int(CB_WIDTH * 0.10), int(CB_WIDTH * 0.90)
    roi_y1, roi_y2 = int(CB_HEIGHT * 0.15), int(CB_HEIGHT * 0.85)
    roi_w, roi_h = roi_x2 - roi_x1, roi_y2 - roi_y1

    # --- 2. 빵판 구멍 자동 인식 (OpenCV) ---
    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    holes_data = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0: continue
        circularity = 4 * np.pi * (area / (perimeter * perimeter))
        if 0.7 < circularity <= 1.2 and 10 < area < 2000:
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                holes_data.append({"x": cx, "y": cy})

    # --- 3. 모델 로드 ---
    weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')
    if not weight_paths:
        print("🚨 [오류] 학습된 모델을 찾을 수 없습니다.")
    else:
        latest_weight_path = max(weight_paths, key=os.path.getctime)
        trained_model = YOLO(latest_weight_path)

        # --- 4. 사진 업로드 및 추론 ---
        print("\n👇 [파일 선택] 버튼을 눌러 실제 빵판 사진을 업로드해주세요!")
        uploaded = files.upload()

        if uploaded:
            real_image_path = list(uploaded.keys())[0]
            real_img = cv2.imread(real_image_path)
            real_h, real_w, _ = real_img.shape

            results = trained_model(real_image_path)
            result = results[0]

            components_data = []

            for i, box in enumerate(result.boxes):
                class_name = trained_model.names[int(box.cls[0].item())].lower()
                xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())

                # 비율 계산
                rel_xmin, rel_ymin = xmin / real_w, ymin / real_h
                rel_xmax, rel_ymax = xmax / real_w, ymax / real_h

                cb_xmin = roi_x1 + int(rel_xmin * roi_w)
                cb_ymin = roi_y1 + int(rel_ymin * roi_h)
                cb_w = int((rel_xmax - rel_xmin) * roi_w)
                cb_h = int((rel_ymax - rel_ymin) * roi_h)

                components_data.append({
                    "id": i,
                    "name": f"{class_name}_{i+1}",
                    "type": class_name,
                    "x": cb_xmin,
                    "y": cb_ymin,
                    "w": cb_w,
                    "h": cb_h
                })

            # --- 5. 배경 이미지를 Base64로 인코딩 ---
            with open(CLEAN_BREADBOARD_IMG, "rb") as image_file:
                encoded_string = base64.b64encode(image_file.read()).decode()
            bg_base64 = f"data:image/png;base64,{encoded_string}"

            json_components = json.dumps(components_data)
            json_holes = json.dumps(holes_data)

            # --- 6. HTML+JS: 드래그 앤 드롭 & 크기 조절 시뮬레이터 ---
            html_code = f"""
            <div style="text-align: center; font-family: sans-serif;">
                <h3 style="color: #333;">🖱️ ↔️ 부품을 잡고 이동하거나, 우측 하단의 <span style="color:red;">빨간 네모</span>를 잡아 크기를 조절하세요!</h3>
                <div style="display: inline-block; position: relative;">
                    <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}"
                            style="border: 2px solid #555; max-width: 100%; height: auto; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);"></canvas>
                </div>
                <br>
                <button onclick="printFinalPositions()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; margin-top: 15px; cursor: pointer; background-color: #4CAF50; color: white; border: none; border-radius: 5px;">✅ 수정 완료 (최종 핀 좌표 출력)</button>
                <pre id="outputConsole" style="text-align: left; background: #222; color: #0f0; padding: 15px; margin-top: 15px; max-height: 250px; overflow-y: auto; border-radius: 5px; font-size: 14px; width: 80%; margin-left: auto; margin-right: auto;"></pre>
            </div>

            <script>
                const canvas = document.getElementById('breadboardCanvas');
                const ctx = canvas.getContext('2d');
                const outputConsole = document.getElementById('outputConsole');

                let items = {json_components};
                let holes = {json_holes};
                let bgImg = new Image();
                bgImg.src = "{bg_base64}";

                let isDragging = false;
                let isResizing = false;
                let dragTarget = null;
                let offsetX, offsetY;
                const HANDLE_SIZE = 12; // 크기 조절 핸들 크기

                bgImg.onload = () => draw();

                // 가장 가까운 구멍 찾기 (자석 기능)
                function getNearestHole(px, py) {{
                    let minDist = Infinity;
                    let nearest = null;
                    holes.forEach(h => {{
                        let d = Math.hypot(h.x - px, h.y - py);
                        if (d < minDist) {{
                            minDist = d;
                            nearest = h;
                        }}
                    }});
                    return nearest;
                }}

                function draw() {{
                    ctx.clearRect(0, 0, canvas.width, canvas.height);
                    ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);

                    items.forEach(item => {{
                        // 몸통 그리기 (마젠타색, 가시성 개선)
                        ctx.fillStyle = "rgba(255, 0, 255, 0.2)";
                        ctx.strokeStyle = "rgba(255, 0, 255, 1)";
                        ctx.lineWidth = 2;
                        ctx.fillRect(item.x, item.y, item.w, item.h);
                        ctx.strokeRect(item.x, item.y, item.w, item.h);

                        // 이름 텍스트
                        ctx.fillStyle = "#8B008B"; // 진한 보라색
                        ctx.font = "bold 16px Arial";
                        ctx.fillText(item.name, item.x, item.y - 8);

                        // 🌟 크기 조절 핸들(빨간색 네모) 우측 하단에 그리기
                        ctx.fillStyle = "red";
                        ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);

                        // 핀(다리) 그리기
                        drawPins(item);
                    }});
                }}

                function drawPins(item) {{
                    let pins = [];
                    // 비율과 크기에 맞춰 핀 위치 동적 계산
                    if (['fet', 'mosfet'].includes(item.type)) {{
                        let mid_x = item.x + (item.w / 2);
                        pins.push({{x: mid_x, y: item.y, color: "blue"}}); // S
                        pins.push({{x: mid_x, y: item.y + (item.h / 2), color: "green"}}); // D
                        pins.push({{x: mid_x, y: item.y + item.h, color: "red"}}); // G
                    }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{
                        if (item.w > item.h) {{
                            let mid_y = item.y + (item.h / 2);
                            pins.push({{x: item.x, y: mid_y, color: "red"}});
                            pins.push({{x: item.x + item.w, y: mid_y, color: "red"}});
                        }} else {{
                            let mid_x = item.x + (item.w / 2);
                            pins.push({{x: mid_x, y: item.y, color: "red"}});
                            pins.push({{x: mid_x, y: item.y + item.h, color: "red"}});
                        }}
                    }} else {{
                        pins.push({{x: item.x, y: item.y + item.h, color: "red"}});
                        pins.push({{x: item.x + item.w, y: item.y + item.h, color: "red"}});
                    }}

                    // 화면에 핀 찍기 및 가장 가까운 구멍과 선 연결 (스냅 시각화)
                    pins.forEach(pin => {{
                        let nearest = getNearestHole(pin.x, pin.y);

                        // 다리에서 진짜 꽂히는 구멍까지 점선 그리기
                        if (nearest) {{
                            ctx.beginPath();
                            ctx.moveTo(pin.x, pin.y);
                            ctx.lineTo(nearest.x, nearest.y);
                            ctx.strokeStyle = "rgba(0,0,0,0.5)";
                            ctx.setLineDash([5, 5]);
                            ctx.stroke();
                            ctx.setLineDash([]);

                            // 구멍에 자석처럼 붙은 위치(정답 좌표)에 검은 테두리 강조
                            ctx.beginPath();
                            ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false);
                            ctx.strokeStyle = "black";
                            ctx.lineWidth = 2;
                            ctx.stroke();
                        }}

                        // 몸통 끝의 핀 위치 그리기
                        ctx.beginPath();
                        ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false);
                        ctx.fillStyle = pin.color;
                        ctx.fill();
                        ctx.strokeStyle = 'white';
                        ctx.stroke();
                    }});
                }}

                function getMousePos(evt) {{
                    const rect = canvas.getBoundingClientRect();
                    const scaleX = canvas.width / rect.width;
                    const scaleY = canvas.height / rect.height;
                    return {{
                        x: (evt.clientX - rect.left) * scaleX,
                        y: (evt.clientY - rect.top) * scaleY
                    }};
                }}

                canvas.addEventListener('mousedown', function(e) {{
                    const pos = getMousePos(e);
                    // 레이어가 겹쳤을 때를 위해 역순 탐색
                    for (let i = items.length - 1; i >= 0; i--) {{
                        let item = items[i];

                        // 1. 크기 조절 핸들(우측 하단)을 클릭했는지 먼저 확인
                        if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w &&
                            pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{
                            isResizing = true;
                            dragTarget = item;
                            canvas.style.cursor = 'nwse-resize';
                            return;
                        }}

                        // 2. 몸통을 클릭했는지 확인 (이동)
                        if (pos.x >= item.x && pos.x <= item.x + item.w &&
                            pos.y >= item.y && pos.y <= item.y + item.h) {{
                            isDragging = true;
                            dragTarget = item;
                            offsetX = pos.x - item.x;
                            offsetY = pos.y - item.y;
                            canvas.style.cursor = 'grabbing';
                            return;
                        }}
                    }}
                }});

                canvas.addEventListener('mousemove', function(e) {{
                    const pos = getMousePos(e);

                    // 마우스 커서 모양 변경 로직 (핸들 위에 올라가면 화살표로)
                    let onHandle = false;
                    for (let i = items.length - 1; i >= 0; i--) {{
                        let item = items[i];
                        if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w &&
                            pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{
                            onHandle = true;
                            break;
                        }}
                    }}
                    if (!isDragging && !isResizing) {{
                        canvas.style.cursor = onHandle ? 'nwse-resize' : 'default';
                    }}

                    // 드래그 또는 리사이즈 처리
                    if (isDragging && dragTarget) {{
                        dragTarget.x = pos.x - offsetX;
                        dragTarget.y = pos.y - offsetY;
                        draw();
                    }} else if (isResizing && dragTarget) {{
                        // 크기 조절 시 최소 크기 제한(20px)
                        let newW = pos.x - dragTarget.x;
                        let newH = pos.y - dragTarget.y;
                        if (newW > 20) dragTarget.w = newW;
                        if (newH > 20) dragTarget.h = newH;
                        draw();
                    }}
                }});

                canvas.addEventListener('mouseup', function(e) {{
                    isDragging = false;
                    isResizing = false;
                    dragTarget = null;
                    canvas.style.cursor = 'default';
                    draw(); // 마우스를 뗄 때 한 번 더 갱신하여 점선 렌더링
                }});

                canvas.addEventListener('mouseleave', function(e) {{
                    isDragging = false;
                    isResizing = false;
                    dragTarget = null;
                    canvas.style.cursor = 'default';
                    draw();
                }});

                function printFinalPositions() {{
                    let out = "🌟 [최종 수정된 핀(Pin) 빵판 구멍 좌표]\\n\\n";
                    items.forEach(item => {{
                        out += `[${{item.name}}]\\n`;

                        let p1, p2, p3;
                        if (['fet', 'mosfet'].includes(item.type)) {{
                            p1 = getNearestHole(item.x + item.w/2, item.y);
                            p2 = getNearestHole(item.x + item.w/2, item.y + item.h/2);
                            p3 = getNearestHole(item.x + item.w/2, item.y + item.h);
                            out += ` - Source (파랑): X(${{p1.x}}), Y(${{p1.y}})\\n`;
                            out += ` - Drain (초록): X(${{p2.x}}), Y(${{p2.y}})\\n`;
                            out += ` - Gate (빨강): X(${{p3.x}}), Y(${{p3.y}})\\n\\n`;
                        }} else {{
                            if (item.w > item.h) {{
                                p1 = getNearestHole(item.x, item.y + item.h/2);
                                p2 = getNearestHole(item.x + item.w, item.y + item.h/2);
                            }} else {{
                                p1 = getNearestHole(item.x + item.w/2, item.y);
                                p2 = getNearestHole(item.x + item.w/2, item.y + item.h);
                            }}
                            out += ` - Pin1: X(${{p1.x}}), Y(${{p1.y}})\\n`;
                            out += ` - Pin2: X(${{p2.x}}), Y(${{p2.y}})\\n\\n`;
                        }}
                    }});
                    outputConsole.innerText = out;
                }}
            </script>
            """

            display(HTML(html_code))

import cv2

import numpy as np

import glob

import os

import base64

import json

from ultralytics import YOLO

from google.colab import files

from IPython.display import HTML, display



# --- 1. 배경 이미지 로드 ---

CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'

cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)



if cb_canvas is None:

    print("🚨 [오류] 'clean_breadboard.png' 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드 해주세요.")

else:

    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape

    roi_x1, roi_x2 = int(CB_WIDTH * 0.10), int(CB_WIDTH * 0.90)

    roi_y1, roi_y2 = int(CB_HEIGHT * 0.15), int(CB_HEIGHT * 0.85)

    roi_w, roi_h = roi_x2 - roi_x1, roi_y2 - roi_y1



    # --- 2. [수정됨] 훨씬 관대한 빵판 구멍 자동 인식 ---

    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV) # 어두운 구멍을 더 잘 찾도록 수정

    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)



    holes_data = []

    for cnt in contours:

        x, y, w, h = cv2.boundingRect(cnt)

        area = w * h

        # 모양이 살짝 찌그러져도 구멍으로 인식하도록 관대한 조건 적용

        if 5 < area < 2500 and 0.4 < w/h < 2.5:

            holes_data.append({"x": x + w//2, "y": y + h//2, "net_id": None})



    if not holes_data:

        print("🚨 [오류] 그림 파일에서 구멍을 전혀 찾지 못했습니다!")

    else:

        # --- 3. 🌟 [완전 개편] 4구역(Zone) 기반 강력한 Net 그룹화 알고리즘 ---

        x_coords = [h['x'] for h in holes_data]

        min_x, max_x = min(x_coords), max(x_coords)

        span_x = max_x - min_x



        # 빵판의 4개 구역을 비율로 자동 분할

        Z1_END = min_x + span_x * 0.18    # 왼쪽 전원부 (+, -)

        Z2_END = min_x + span_x * 0.48    # 왼쪽 메인부 (a~e)

        Z3_START = min_x + span_x * 0.52  # 오른쪽 메인부 (f~j)

        Z4_START = min_x + span_x * 0.82  # 오른쪽 전원부 (+, -)



        # 오차 범위를 매우 넉넉하게 설정 (전체 폭의 1.5%)

        TOL = span_x * 0.015



        net_map = {i: i for i in range(len(holes_data))}

        def find(i):

            if net_map[i] == i: return i

            net_map[i] = find(net_map[i])

            return net_map[i]

        def union(i, j):

            root_i, root_j = find(i), find(j)

            if root_i != root_j: net_map[root_i] = root_j



        # 구역별 맞춤형 연결 로직

        for i in range(len(holes_data)):

            for j in range(i+1, len(holes_data)):

                h1, h2 = holes_data[i], holes_data[j]

                dx, dy = abs(h1['x'] - h2['x']), abs(h1['y'] - h2['y'])



                # 1번 구멍의 구역 찾기

                if h1['x'] < Z1_END: z1 = 1

                elif h1['x'] < Z2_END: z1 = 2

                elif h1['x'] > Z4_START: z1 = 4

                elif h1['x'] >= Z3_START: z1 = 3

                else: z1 = 0



                # 2번 구멍의 구역 찾기

                if h2['x'] < Z1_END: z2 = 1

                elif h2['x'] < Z2_END: z2 = 2

                elif h2['x'] > Z4_START: z2 = 4

                elif h2['x'] >= Z3_START: z2 = 3

                else: z2 = 0



                # 같은 구역 안에 있는 구멍들만 묶습니다!

                if z1 == z2 and z1 != 0:

                    if z1 == 2 or z1 == 3: # [메인 구역] 가로로 인접하면 묶음 (Y가 같아야 함)

                        if dy < TOL: union(i, j)

                    elif z1 == 1 or z1 == 4: # [전원 구역] 세로로 인접하면 묶음 (X가 같아야 함)

                        if dx < TOL: union(i, j)



        # Net 이름 부여

        net_dict = {}

        net_counter = 1

        for i in range(len(holes_data)):

            root = find(i)

            if root not in net_dict:

                net_dict[root] = f"NET_{net_counter:03d}"

                net_counter += 1

            holes_data[i]['net_id'] = net_dict[root]



        print(f"🎯 인식된 구멍: {len(holes_data)}개 / 생성된 회로 라인(Net): {net_counter-1}개 (성공!)")



        # --- 4. 모델 추론 ---

        weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')

        if not weight_paths:

            print("🚨 [오류] 학습된 모델을 찾을 수 없습니다.")

        else:

            trained_model = YOLO(max(weight_paths, key=os.path.getctime))



            print("\n👇 [파일 선택] 버튼을 눌러 실제 빵판 사진을 업로드해주세요!")

            uploaded = files.upload()



            if uploaded:

                real_image_path = list(uploaded.keys())[0]

                real_img = cv2.imread(real_image_path)

                real_h, real_w, _ = real_img.shape



                results = trained_model(real_image_path)

                result = results[0]



                components_data = []

                for i, box in enumerate(result.boxes):

                    class_name = trained_model.names[int(box.cls[0].item())].lower()

                    xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())



                    rel_xmin, rel_ymin = xmin / real_w, ymin / real_h

                    rel_xmax, rel_ymax = xmax / real_w, ymax / real_h



                    cb_xmin = roi_x1 + int(rel_xmin * roi_w)

                    cb_ymin = roi_y1 + int(rel_ymin * roi_h)

                    cb_w = int((rel_xmax - rel_xmin) * roi_w)

                    cb_h = int((rel_ymax - rel_ymin) * roi_h)



                    components_data.append({

                        "id": i, "name": f"{class_name}_{i+1}", "type": class_name,

                        "x": cb_xmin, "y": cb_ymin, "w": cb_w, "h": cb_h

                    })



                with open(CLEAN_BREADBOARD_IMG, "rb") as image_file:

                    encoded_string = base64.b64encode(image_file.read()).decode()

                bg_base64 = f"data:image/png;base64,{encoded_string}"



                json_components = json.dumps(components_data)

                json_holes = json.dumps(holes_data)



                # --- 5. HTML+JS 시뮬레이터 (선 그리기 로직 개선) ---

                html_code = f"""

                <div style="text-align: center; font-family: sans-serif;">

                    <h3 style="color: #333;">💡 이제 <span style="color: #00AA00; font-weight:bold;">초록색 연결망</span>이 완벽하게 보일 겁니다!</h3>

                    <p>같은 초록색 선 위에 부품들의 다리를 꽂고 추출 버튼을 눌러보세요.</p>

                    <div style="display: inline-block; position: relative;">

                        <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}"

                                style="border: 2px solid #555; max-width: 100%; height: auto; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);"></canvas>

                    </div>

                    <br>

                    <button onclick="generateNetlist()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; margin-top: 15px; cursor: pointer; background-color: #007BFF; color: white; border: none; border-radius: 5px;">✅ 넷리스트 추출하기</button>

                    <pre id="outputConsole" style="text-align: left; background: #222; color: #0f0; padding: 15px; margin-top: 15px; max-height: 400px; overflow-y: auto; border-radius: 5px; font-size: 15px; width: 80%; margin-left: auto; margin-right: auto; line-height: 1.6;"></pre>

                </div>



                <script>

                    const canvas = document.getElementById('breadboardCanvas');

                    const ctx = canvas.getContext('2d');

                    const outputConsole = document.getElementById('outputConsole');



                    let items = {json_components};

                    let holes = {json_holes};

                    let bgImg = new Image();

                    bgImg.src = "{bg_base64}";



                    let isDragging = false, isResizing = false;

                    let dragTarget = null, offsetX, offsetY;

                    const HANDLE_SIZE = 12;



                    let netGroups = {{}};

                    holes.forEach(h => {{

                        if (!netGroups[h.net_id]) netGroups[h.net_id] = [];

                        netGroups[h.net_id].push(h);

                    }});



                    bgImg.onload = () => draw();



                    function getNearestHole(px, py) {{

                        let minDist = Infinity, nearest = null;

                        holes.forEach(h => {{

                            let d = Math.hypot(h.x - px, h.y - py);

                            if (d < minDist) {{ minDist = d; nearest = h; }}

                        }});

                        return nearest;

                    }}



                    function draw() {{

                        ctx.clearRect(0, 0, canvas.width, canvas.height);

                        ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);



                        // 🌟 빵판 내부 연결 실선 그리기 (그리기 로직 개선)

                        ctx.lineWidth = 6; // 선을 더 두껍게

                        ctx.strokeStyle = "rgba(0, 220, 50, 0.6)"; // 더 밝은 초록색

                        ctx.lineCap = "round";

                        ctx.lineJoin = "round";



                        for (let net in netGroups) {{

                            let hList = netGroups[net];

                            if (hList.length > 1) {{

                                // 세로줄인지 가로줄인지 파악하여 정렬 방향 결정

                                let dx = Math.abs(hList[0].x - hList[hList.length-1].x);

                                let dy = Math.abs(hList[0].y - hList[hList.length-1].y);



                                if (dx > dy) hList.sort((a, b) => a.x - b.x); // 가로 정렬

                                else hList.sort((a, b) => a.y - b.y); // 세로 정렬



                                ctx.beginPath();

                                ctx.moveTo(hList[0].x, hList[0].y);

                                for(let i=1; i<hList.length; i++) {{

                                    ctx.lineTo(hList[i].x, hList[i].y);

                                }}

                                ctx.stroke();

                            }}

                        }}



                        // 부품 그리기

                        items.forEach(item => {{

                            ctx.fillStyle = "rgba(255, 0, 255, 0.15)";

                            ctx.strokeStyle = "rgba(255, 0, 255, 1)";

                            ctx.lineWidth = 2;

                            ctx.fillRect(item.x, item.y, item.w, item.h);

                            ctx.strokeRect(item.x, item.y, item.w, item.h);



                            ctx.fillStyle = "#8B008B";

                            ctx.font = "bold 16px Arial";

                            ctx.fillText(item.name, item.x, item.y - 8);



                            ctx.fillStyle = "red";

                            ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);



                            drawPins(item);

                        }});

                    }}



                    function drawPins(item) {{

                        let pins = getPinCoords(item);

                        pins.forEach(pin => {{

                            let nearest = getNearestHole(pin.x, pin.y);

                            if (nearest) {{

                                ctx.beginPath();

                                ctx.moveTo(pin.x, pin.y); ctx.lineTo(nearest.x, nearest.y);

                                ctx.strokeStyle = "rgba(0,0,0,0.5)"; ctx.setLineDash([5, 5]); ctx.stroke(); ctx.setLineDash([]);



                                ctx.beginPath();

                                ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false);

                                ctx.strokeStyle = "black"; ctx.lineWidth = 2; ctx.stroke();

                            }}



                            ctx.beginPath();

                            ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false);

                            ctx.fillStyle = pin.color; ctx.fill(); ctx.strokeStyle = 'white'; ctx.stroke();

                        }});

                    }}



                    function getPinCoords(item) {{

                        let pins = [];

                        if (['fet', 'mosfet'].includes(item.type)) {{

                            let mid_x = item.x + (item.w / 2);

                            pins.push({{name: "Source", x: mid_x, y: item.y, color: "blue"}});

                            pins.push({{name: "Drain", x: mid_x, y: item.y + (item.h / 2), color: "green"}});

                            pins.push({{name: "Gate", x: mid_x, y: item.y + item.h, color: "red"}});

                        }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{

                            if (item.w > item.h) {{

                                let mid_y = item.y + (item.h / 2);

                                pins.push({{name: "Pin1", x: item.x, y: mid_y, color: "red"}});

                                pins.push({{name: "Pin2", x: item.x + item.w, y: mid_y, color: "red"}});

                            }} else {{

                                let mid_x = item.x + (item.w / 2);

                                pins.push({{name: "Pin1", x: mid_x, y: item.y, color: "red"}});

                                pins.push({{name: "Pin2", x: mid_x, y: item.y + item.h, color: "red"}});

                            }}

                        }} else {{

                            pins.push({{name: "Pin1", x: item.x, y: item.y + item.h, color: "red"}});

                            pins.push({{name: "Pin2", x: item.x + item.w, y: item.y + item.h, color: "red"}});

                        }}

                        return pins;

                    }}



                    function getMousePos(evt) {{

                        const rect = canvas.getBoundingClientRect();

                        const scaleX = canvas.width / rect.width; const scaleY = canvas.height / rect.height;

                        return {{ x: (evt.clientX - rect.left) * scaleX, y: (evt.clientY - rect.top) * scaleY }};

                    }}



                    canvas.addEventListener('mousedown', function(e) {{

                        const pos = getMousePos(e);

                        for (let i = items.length - 1; i >= 0; i--) {{

                            let item = items[i];

                            if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w && pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{

                                isResizing = true; dragTarget = item; canvas.style.cursor = 'nwse-resize'; return;

                            }}

                            if (pos.x >= item.x && pos.x <= item.x + item.w && pos.y >= item.y && pos.y <= item.y + item.h) {{

                                isDragging = true; dragTarget = item; offsetX = pos.x - item.x; offsetY = pos.y - item.y; canvas.style.cursor = 'grabbing'; return;

                            }}

                        }}

                    }});



                    canvas.addEventListener('mousemove', function(e) {{

                        const pos = getMousePos(e);

                        if (isDragging && dragTarget) {{ dragTarget.x = pos.x - offsetX; dragTarget.y = pos.y - offsetY; draw(); }}

                        else if (isResizing && dragTarget) {{

                            let newW = pos.x - dragTarget.x; let newH = pos.y - dragTarget.y;

                            if (newW > 20) dragTarget.w = newW; if (newH > 20) dragTarget.h = newH;

                            draw();

                        }}

                    }});



                    canvas.addEventListener('mouseup', function(e) {{

                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default'; draw();

                    }});



                    canvas.addEventListener('mouseleave', function(e) {{

                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default'; draw();

                    }});



                    function generateNetlist() {{

                        let netlistMap = {{}};



                        items.forEach(item => {{

                            let pins = getPinCoords(item);

                            pins.forEach(pin => {{

                                let nearestHole = getNearestHole(pin.x, pin.y);

                                if (nearestHole && nearestHole.net_id) {{

                                    if (!netlistMap[nearestHole.net_id]) netlistMap[nearestHole.net_id] = [];

                                    netlistMap[nearestHole.net_id].push(`${{item.name}}(${{pin.name}})`);

                                }}

                            }});

                        }});



                        let out = "🔌 [최종 추출된 전기적 회로 넷리스트]\\n\\n";

                        let validConnections = false;



                        out += "🔗 [상호 연결된 노드]\\n";

                        for (let net in netlistMap) {{

                            if (netlistMap[net].length > 1) {{

                                out += `  - ${{net}} :  ${{netlistMap[net].join("  ↔  ")}}\\n`;

                                validConnections = true;

                            }}

                        }}



                        if (!validConnections) out += "  (아직 연결된 부품이 없습니다. 초록색 선을 공유하도록 다리를 놔주세요!)\\n";



                        out += "\\n📌 [단일 연결 (혼자 꽂힌 핀)]\\n";

                        for (let net in netlistMap) {{

                            if (netlistMap[net].length === 1) {{

                                out += `  - ${{net}} :  ${{netlistMap[net][0]}}\\n`;

                            }}

                        }}



                        outputConsole.innerText = out;

                    }}

                </script>

                """



                display(HTML(html_code))
